In [2]:
from typing import List, Tuple, Literal, Dict, Any
import os
import numpy as np
import mne
from data_reader_2023 import *
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy import fftpack
# import h5py 
import random
import shutil
from scipy.signal import iirnotch, lfilter
from scipy.fft import rfft, rfftfreq
import pandas as pd

In [3]:
%load_ext autoreload
%autoreload 2
# Recarga automáticamente todos los módulos importados.

## Define Constants and Key Variables

In [4]:
########### Constants
# Define bandpass filter constants
lowcut: float = 0.5
highcut: float = 120.0
fs: int = 1024
resampleFS: int = 250

# Cada filtro devuelve dos arrays (b, a) de coeficientes
notch_1_b: np.ndarray
notch_1_a: np.ndarray
notch_1_b, notch_1_a = iirnotch(1.0, Q=30.0, fs=resampleFS)


notch_60_b: np.ndarray
notch_60_a: np.ndarray
notch_60_b, notch_60_a = iirnotch(60.0, Q=30.0, fs=resampleFS)
# notch_100_b, notch_100_a = iirnotch(100, Q=30, fs=250)

# Define segment interval length in sec
segment_interval: int = 4
print('Segment Interval:', segment_interval)

# Seizure types
binary_classifier_flag: bool = True

# seizure_types = ['fnsz', 'gnsz', 'cpsz', 'absz', 'tnsz', 'tcsz', 'bckg']
if binary_classifier_flag:
    seizure_types: List[str] = ['bckg', 'seizure']
    seizure_session_downsampling_ratio: List[float] = [1.0, 1.0]
    seizure_overlapping_ratio: List[float] = [0.0, 0.75]
else:
    seizure_types: List[str] = ['fnsz', 'gnsz', 'cpsz', 'bckg']
    seizure_session_downsampling_ratio: List[float] = [1.0, 1.0, 1.0, 1.0]
    seizure_overlapping_ratio: List[float] = [0.75, 0.75, 0.75, 0.0]

Segment Interval: 4


In [5]:
########## Data path
train_val_root: str = os.path.join(
    'dataset', 'tuh_eeg_seizure', 'v2.0.3', 'edf', 'train'
)
test_root: str = os.path.join(
    'dataset', 'tuh_eeg_seizure', 'v2.0.3', 'edf', 'dev'
)
# test_root: str = os.path.join('/datadrive', 'TUSZ_2023', 'edf', 'eval')

print('Train/Val root: ', train_val_root)
print('Test root: ', test_root)

if binary_classifier_flag:
    save_root: str = os.path.join(
        'dataset',
        'tuh_eeg_seizure',
        'v2.0.3',
        'TUSZ_processed_binary_individual_segments'
    )
else:
    save_root: str = os.path.join(
        'dataset',
        'tuh_eeg_seizure',
        'v2.0.3',
        'TUSZ_processed_multiclass_individual_segments'
    )

print('Save root:', save_root)

if not os.path.exists(save_root):
    os.mkdir(save_root)

# Modos de datos permitidos
DataMode = Literal['small', 'large', 'tiny', 'full']
data_mode: DataMode = 'full'

Train/Val root:  dataset/tuh_eeg_seizure/v2.0.3/edf/train
Test root:  dataset/tuh_eeg_seizure/v2.0.3/edf/dev
Save root: dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments


## To delete previous data repo when running a new experiment

In [6]:
## To delete previous data repo when running a new experiment
## This is to avoid concatenating duplicate data to the existing .npy files

# Construye el path del directorio para este experimento
segment_folder: str = os.path.join(
    save_root,
    f"segment_interval_{segment_interval}_sec"
)
print('Segment folder:', segment_folder)

# Si no existe, lo creamos (experimento nuevo)
if not os.path.exists(segment_folder):
    print('Creating new segment folder:', segment_folder)
    os.mkdir(segment_folder)
else:
    # Si ya existe, borramos todo su contenido para evitar duplicados
    # al concatenar archivos .npy en ejecuciones sucesivas
    print('Deleting existing segment folder:', segment_folder)
    filenames: List[str] = os.listdir(segment_folder)
    for filename in filenames:
        file_path: str = os.path.join(segment_folder, filename)
        try:
            # Si es archivo o enlace simbólico, lo eliminamos
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            # Si es un directorio, lo borramos recursivamente
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            # Captura cualquier error en la eliminación y lo informa
            error_msg: str = f"Failed to delete {file_path}. Reason: {e}"
            print(error_msg)

Segment folder: dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments/segment_interval_4_sec
Deleting existing segment folder: dataset/tuh_eeg_seizure/v2.0.3/TUSZ_processed_binary_individual_segments/segment_interval_4_sec


In [7]:
# Obtener rutas de sesión, lista de pacientes y conteo de tipos de referencia
train_val_paths: List[str]
train_val_patients: List[str]
train_val_reference_type_count: Dict[str, int]
train_val_paths, train_val_patients, train_val_reference_type_count = get_all_TUSZ_2023_session_paths(train_val_root)

print('Total sessions:', len(train_val_paths))
print('Train and val patients:', len(train_val_patients))
print('Train and val reference type count:', train_val_reference_type_count)

# val_paths, val_patients = get_all_TUSZ_2023_session_paths(val_root)
# print('Total sessions: ', len(val_paths))
# print('Total patients: ', len(val_patients))

# Rutas y pacientes de test
test_paths: List[str]
test_patients: List[str]
test_reference_type_count: Dict[str, int]
test_paths, test_patients, test_reference_type_count = get_all_TUSZ_2023_session_paths(test_root)

print('Test sessions:', len(test_paths))
print('Test patients:', len(test_patients))
print('Test reference type count:', test_reference_type_count)

# Unión de todas las rutas de sesión
all_paths: List[str] = train_val_paths + test_paths
print('All sessions:', len(all_paths))

# Estructuras para almacenar datos por tipo de clase
# Cada sublista corresponderá a uno de los labels en seizure_types
training_data: List[List[Any]] = [[] for _ in seizure_types]
validation_data: List[List[Any]] = [[] for _ in seizure_types]
testing_data: List[List[Any]] = [[] for _ in seizure_types]

# Fijamos semilla para reproducibilidad y mezclamos pacientes
random.seed(42)
random.shuffle(train_val_patients)


# División 80/20 de pacientes entre entrenamiento y validación
train_patients: List[str] = train_val_patients[:int(len(train_val_patients) * 0.8)]
val_patients: List[str] = train_val_patients[int(len(train_val_patients) * 0.8):]

print("---------------------------------------")
print('Train patients:', len(train_patients))
print('Val patients:', len(val_patients))


# Total sessions:  4664
# Train and val patients:  579
# Test sessions:  1832
# Test patients:  53
# All sessions:  6496

Total sessions: 4664
Train and val patients: 579
Train and val reference type count: {'01_tcp_ar': 683, '02_tcp_le': 324, '03_tcp_ar_a': 168}
Test sessions: 1832
Test patients: 53
Test reference type count: {'01_tcp_ar': 268, '03_tcp_ar_a': 36, '02_tcp_le': 38}
All sessions: 6496
---------------------------------------
Train patients: 463
Val patients: 116


In [8]:
edades: List[int] = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
random.seed(42)
random.shuffle(edades)
edades_train: List[int] = edades[:7]
edades_val: List[int] = edades[7:]
print(edades_train)
print(edades_val)

[8, 4, 3, 9, 6, 7, 10]
[5, 1, 2]


In [9]:
# Probar lectura de un archivo CSV
data_path: str = 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.edf'
print('Data path:', data_path)

label_file = data_path[:-4] + '.csv'
print('[TEST]: ', data_path[:-4])
print('label_file:', label_file)
# Los .csv del datset de TUSZ 2023 tienen un encabezado de 5 filas
df = pd.read_csv(label_file, skiprows=5, header=0)
df

Data path: dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.edf
[TEST]:  dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000
label_file: dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.csv


,channel,start_time,stop_time,label,confidence
0,FP1-F7,0.0,1205.0,bckg,1.0
1,F7-T3,0.0,1205.0,bckg,1.0
2,T3-T5,0.0,1205.0,bckg,1.0
3,T5-O1,0.0,1205.0,bckg,1.0
4,FP2-F8,0.0,1205.0,bckg,1.0
5,F8-T4,0.0,1205.0,bckg,1.0
6,T4-T6,0.0,1205.0,bckg,1.0
7,T6-O2,0.0,1205.0,bckg,1.0
8,A1-T3,0.0,1205.0,bckg,1.0
9,T3-C3,0.0,1205.0,bckg,1.0


In [10]:
russellTestPaths: List[str] = [
    'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.edf',
    'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t001.edf',
    'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t002.edf'
]

for data_path in russellTestPaths:
    print(data_path)
    patient = data_path.split('train/')[1].split('/')[0]
    print('\tPatient:', patient)

    reference_type = data_path.split('train/')[1].split('/')[2]
    print('\tReference type:', reference_type)
    # temp = data_path.split('train/')[1].split('/')
    # print("\tTemp: ", temp)

dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t000.edf
	Patient: aaaaaacz
	Reference type: 01_tcp_ar
dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t001.edf
	Patient: aaaaaacz
	Reference type: 01_tcp_ar
dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaacz/s003_2010/01_tcp_ar/aaaaaacz_s003_t002.edf
	Patient: aaaaaacz
	Reference type: 01_tcp_ar


In [28]:

# dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf
# dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaedy/s002_2004/02_tcp_le/aaaaaedy_s002_t001.edf
# dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaedy/s003_2004/02_tcp_le/aaaaaedy_s003_t000.edf
count_session:int = 0
sample_paths = train_val_paths[:10]
print("Sample paths: ", sample_paths)

# for data_path in train_val_paths:

contador_02_tcp_le: int = 0
for data_path in sample_paths:
    patient:str = data_path.split('train/')[1].split('/')[0]
    reference_type:str = data_path.split('train/')[1].split('/')[2]
    if reference_type == '02_tcp_le':
        contador_02_tcp_le += 1
        continue
    patient_session:str = data_path.split('train/')[1].split('/')[-1][:-4]
    # patient_session = patient+'_'+data_path.split('01_tcp_ar')[1][14:18]

    print("after if: ", reference_type)

    # --------------------------------------------------
    # 3) Set 'flag_train_val_test' train/val/test
    # --------------------------------------------------
    if patient in train_patients:
        # continue
        flag_train_val_test:str = 'train'
    elif patient in val_patients:
        # continue
        flag_train_val_test:str = 'val'
    elif patient in test_patients:
        flag_train_val_test:str = 'test'
    else:
        print('Patient not found in train, val, or test lists:', patient)
        continue;
    # if data_path != '/datadrive/yuandaData/edf/train/01_tcp_ar/091/00009104/s011_2014_09_29/00009104_s011_t000.edf':
    #     continue

    count_session += 1
    # print('Patient is: ', patient)
    # print('Patient belongs to: ', flag_train_val_test)
    # print('Patient session is: ', patient_session)
    # break
    
    ### Read the raw signals
    # --------------------------------------------------
    # 4) Carga de la señal cruda EDF
    # --------------------------------------------------
    raw: mne.io.BaseRaw = mne.io.read_raw_edf(data_path, verbose='warning')
    # print(raw.info)    
    thisFS: int = int(raw.info['sfreq'])
    # if thisFS != 250:
    #     continue

    # --------------------------------------------------
    # 5) Extracción de canales
    # --------------------------------------------------
    flag_wrong: bool
    signals: np.ndarray
    flag_wrong, signals = get_channels_from_raw(raw)
    if flag_wrong:
        # print('flag_wrong:', flag_wrong)
        continue
    
    # make_a_sample_plot_from_array(signals, thisFS)
    # make_a_frequency_plot_from_array(signals, thisFS)
    
    ### Butter bandpass filter
    # --------------------------------------------------
    # 6) Filtrado bandpass + notch
    # --------------------------------------------------
    filtered_signals: List[np.ndarray] = []
    for i in range(signals.shape[0]):
        bandpass_filtered_signal: np.ndarray = butter_bandpass_filter(
            signals[i, :], lowcut, highcut, fs, order=3
        )
        filtered_1_signal: np.ndarray = lfilter(notch_1_b, notch_1_a, bandpass_filtered_signal)
        filtered_60_signal: np.ndarray = lfilter(notch_60_b, notch_60_a, filtered_1_signal)
        filtered_signals.append(filtered_60_signal)

    # make_a_filtered_plot_for_comparison(signals, filtered_signals, thisFS)
    # plot_signal_in_frequency(signals[0], filtered_signals[0], thisFS)
    # # print('Sampling Frequency is: ', thisFS)

    # break
    ### Resampling
    # resampled_signals = []
    # --------------------------------------------------
    # 7) Remuestreo
    # --------------------------------------------------
    resampled_signals: List[np.ndarray]
    if thisFS == resampleFS:
        resampled_signals = filtered_signals[:]
    else:
        resampled_signals = resample_data_in_each_channel(filtered_signals, thisFS, resampleFS)
    

    # --------------------------------------------------
    # 8) Leer siempre el CSV multiclase (todas las etiquetas)
    # --------------------------------------------------
    ### Read tse labels- OBSOLETE
    # labels = []
    # tseFile = data_path[:-4] + '.tse'
    # with open(tseFile,'r') as tseReader:
    #     rawText = tseReader.readlines()[2:]
    #     seizPeriods = []
    #     for item in rawText:
    #         labels.append([int(item.split()[0].split('.')[0]),int(item.split()[1].split('.')[0]),item.split()[2]])
    label_csv: str = data_path[:-4] + '.csv'
    df: pd.DataFrame = pd.read_csv(label_csv, skiprows=5, header=0)
    labels: List[Tuple[int, int, str]] = []
    for _, row in df.iterrows():
        onset: float = float(row['onset'])
        duration: float = float(row['duration'])
        start_s: int = int(onset)
        end_s:   int = int(onset + duration)
        sz_type: str = str(row['type'])
        labels.append((start_s, end_s, sz_type))    
    
    ### Slice signal into segments, and assign proper labels
    # --------------------------------------------------
    # 9) Segmentación (binario o multiclasificación)
    # --------------------------------------------------
    segments: List[List[List[np.ndarray]]]
    if binary_classifier_flag:
        segments = slice_signals_into_binary_segments(
            filtered_signals,
            thisFS,
            labels,
            segment_interval,
            seizure_types,
            seizure_overlapping_ratio
        )
    else:
        segments = slice_signals_into_multiclass_segments(
            filtered_signals,
            thisFS,
            labels,
            segment_interval,
            seizure_types,
            seizure_overlapping_ratio
        )
    
    # segments[:]: list for different seizure labels
    # segments[0][:]: list for different annotation lines within the same annotation file (.tse)
    # segments[0][0][:]: list for different windows of EEG signals
    # segments[0][0][0]: a numpy array of the shape (22, FS*segment_interval)

    # if segments[0]:
    #     break
    
    # --------------------------------------------------
    # 10) Guardado de los .npy
    # --------------------------------------------------
    for i in range(len(segments)):
        if segments[i] and segments[i][0]:
            this_array: List[np.ndarray] = []
            this_labels:str = seizure_types[i]
            for j in range(len(segments[i])):
                if not segments[i][j]:
                    continue
                for k in range(len(segments[i][j])):
                    this_array.append(segments[i][j][k])

            # if not os.path.exists(os.path.join(save_root, 'segment_interval_'+str(segment_interval)+'_sec')):
            #     os.mkdir(os.path.join(save_root, 'segment_interval_'+str(segment_interval)+'_sec'))
            # if not os.path.exists(os.path.join(save_root, 'segment_interval_'+str(segment_interval)+'_sec', flag_train_val_test)):
            #     os.mkdir(os.path.join(save_root, 'segment_interval_'+str(segment_interval)+'_sec', flag_train_val_test))
            # save_folder = os.path.join(save_root, 'segment_interval_'+str(segment_interval)+'_sec', flag_train_val_test, this_labels)
            # if not os.path.exists(save_folder):
            #     os.mkdir(save_folder)
            # save_file = os.path.join(save_folder, patient_session+'.npy')
            folder_base: str = os.path.join(
                save_root,
                f'segment_interval_{segment_interval}_sec',
                flag_train_val_test,
                this_labels
            )
            os.makedirs(folder_base, exist_ok=True)
            save_file: str = os.path.join(folder_base, f'{patient_session}.npy')
            if os.path.isfile(save_file):
                # If the file exists, load the existing data
                existing_data: np.ndarray = np.load(save_file, allow_pickle=True)
                # Append the new data to the existing data
                new_data: np.ndarray = np.concatenate((existing_data, np.array(this_array)))
                # Save the combined data to the file
                np.save(save_file, new_data)
                print(new_data.shape)
            else:
                print(np.array(this_array).shape)
                np.save(save_file, np.array(this_array))


    if data_mode == 'small':                
        if count_session >= 1500:
            break
    elif data_mode == 'tiny':
        if count_session >= 100:
            break
    elif data_mode == 'large':
        if count_session >= 2500:
            break


print("contador_02_tcp_le: ", contador_02_tcp_le)

Sample paths:  ['dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaamtj/s002_2012/01_tcp_ar/aaaaamtj_s002_t000.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaamtj/s001_2012/01_tcp_ar/aaaaamtj_s001_t001.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaamtj/s001_2012/01_tcp_ar/aaaaamtj_s001_t000.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaamtj/s001_2012/01_tcp_ar/aaaaamtj_s001_t002.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s002_2003/02_tcp_le/aaaaaauj_s002_t001.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s003_2003/02_tcp_le/aaaaaauj_s003_t001.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s001_2003/02_tcp_le/aaaaaauj_s001_t001.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaauj/s004_2012/01_tcp_ar/aaaaaauj_s004_t000.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaedy/s002_2004/02_tcp_le/aaaaaedy_s002_t001.edf', 'dataset/tuh_eeg_seizure/v2.0.3/edf/train/aaaaaedy/s003_2004/02_tcp_le/aaaaaedy_s003_t000.edf']
after if:  01_tcp_ar
Som